
## Training AI agent Decision Helper models 



## Closed-Loop Agent Learning (CLAL)


In [1]:

## !pip install scikit-learn


In [2]:

# ================================
# CREWAI + TRAINED TOOL SELECTION
# ================================

from crewai import Agent
from langchain.tools import Tool

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



In [3]:

import os
from crewai_tools import SerperDevTool, BrowserbaseLoadTool, EXASearchTool
from crewai import Agent
from crewai import Task
from crewai import LLM

from langchain_community.llms import Ollama


from crewai import Crew, Process


In [4]:

# -------------------------------
# 1. DEFINE TOOLS
# -------------------------------
def calculator_tool(x: str):
    try:
        return str(eval(x))
    except:
        return "Error in calculation"


In [5]:



def search_tool(x: str):
    return f"Searching for: {x}"


In [6]:

# -------------------------------
# 2. LOGGING WRAPPER
# -------------------------------
logs = []

def logging_tool(tool_name, func):
    
    def wrapper(x):
        logs.append((x, tool_name))
        return func(x)
        
    return wrapper



In [7]:

tools = [
    Tool(
        name="Calculator",
        func=logging_tool("Calculator", calculator_tool),
        description="Use for math operations"
    ),
    Tool(
        name="Search",
        func=logging_tool("Search", search_tool),
        description="Use for general knowledge"
    )
]



In [8]:

# -------------------------------
# 3. CREWAI AGENT (FOR DATA COLLECTION)
# -------------------------------



agent = Agent(
    role='Assistant',
    goal='Solve problems using the correct tool',
    backstory="""You choose tools wisely.""",
    verbose=False,
    tools=tools,
    ## llm=llm_llama
    llm=LLM( model="ollama/llama3.2", base_url="http://localhost:11434" )
)


In [9]:

research_task = Task(
    description="Conduct comprehensive research on the latest fusion energy developments.",
    expected_output="A summary report highlighting key advancements in fusion energy that is really funny, with lots of dad jokes",
    agent=agent,
    ## tools=[ WebsiteSearchTool() ],
    async_execution=True
)


In [10]:

my_crew = Crew(
    agents=[agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True
)


In [11]:

result = my_crew.kickoff()
print(result)


# Agent: Assistant
## Task: Conduct comprehensive research on the latest fusion energy developments.


# Agent: Assistant
## Using tool: Search
## Tool Input: 
"{\"x\": \"fusion energy latest developments\"}"
## Tool Output: 
Searching for: fusion energy latest developments


# Agent: Assistant
## Final Answer: 
fusion energy latest developments are a hot topic. 
The quest to harness the power of the stars is no laughing matter, but someone's gotta do it! 
The pursuit of controlled nuclear fusion has been going on for decades, with scientists working tirelessly to achieve this holy grail of energy production.

Let's start with some of the recent advancements in the field:
The International Thermonuclear Experimental Reactor (ITER) project is currently underway in France, aiming to demonstrate the feasibility of fusion power. 
Meanwhile, private companies like Lockheed Martin and General Fusion are also making waves with their innovative approaches to achieving controlled fusion.
 
Rese

In [12]:

print(f"Raw Output: {result.raw}")


Raw Output: fusion energy latest developments are a hot topic. 
The quest to harness the power of the stars is no laughing matter, but someone's gotta do it! 
The pursuit of controlled nuclear fusion has been going on for decades, with scientists working tirelessly to achieve this holy grail of energy production.

Let's start with some of the recent advancements in the field:
The International Thermonuclear Experimental Reactor (ITER) project is currently underway in France, aiming to demonstrate the feasibility of fusion power. 
Meanwhile, private companies like Lockheed Martin and General Fusion are also making waves with their innovative approaches to achieving controlled fusion.
 
Researchers at MIT have made significant breakthroughs in understanding the behavior of plasmas, which is crucial for achieving sustained fusion reactions. 
And then there's the development of magnetized targets, which could potentially lead to a more efficient and scalable approach to fusion energy.

But

In [13]:

import pandas as pd
df = pd.read_csv("sales.csv")


In [14]:


df


,Date,Revenue,Expenses,Customers
0,2024-01-31,12000,8000,234
1,2024-02-29,10000,8500,201
2,2024-03-31,9300,8700,198
3,2024-04-30,11500,8100,225
4,2024-05-31,9800,7900,210
5,2024-06-30,10400,8000,215


In [15]:

import pandas as pd
from crewai import Agent, Task, Crew

from langchain.tools import tool

# Load CSV
df = pd.read_csv("sales.csv")

@tool
def summarize_csv() -> str:
    """Summarizes the dataframe by showing the first few rows and statistical metrics."""
    summary = df.describe(include='all').to_string()
    preview = df.head().to_string()
    return f"Data Preview:\n{preview}\n\nSummary Statistics:\n{summary}"



analyst = Agent(
    role='CSV Analyst',
    goal='Read the spreadsheet and summarize key metrics',
    backstory="""You're a researcher at a large company.
    You're responsible for analyzing data and providing insights
    to the business.""",
    verbose=False,
    ## llm=llm_llama
    tools=[   summarize_csv  ],
    llm=LLM( model="ollama/llama3.2", base_url="http://localhost:11434" )
)



task = Task(
    description="Summarize the spreadsheet, including any trends or patterns.",
    expected_output="A markdown-style summary report of the dataset including basic statistics and key insights.",
    agent=analyst
)

# === Crew ===
crew = Crew(agents=[analyst], tasks=[task], verbose=True)

if __name__ == "__main__":
    result = crew.kickoff()
    print("\n--- Summary Report ---\n")
    print(result)


2026-04-15 18:03:57,561 - 8608547392 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: CSV Analyst
## Task: Summarize the spreadsheet, including any trends or patterns.


# Agent: CSV Analyst
## Using tool: summarize_csv
## Tool Input: 
"{\"df\": \"data.csv\"}"
## Tool Output: 
Data Preview:
         Date  Revenue  Expenses  Customers
0  2024-01-31    12000      8000        234
1  2024-02-29    10000      8500        201
2  2024-03-31     9300      8700        198
3  2024-04-30    11500      8100        225
4  2024-05-31     9800      7900        210

Summary Statistics:
              Date       Revenue    Expenses   Customers
count            6      6.000000     6.00000    6.000000
unique           6           NaN         NaN         NaN
top     2024-01-31           NaN         NaN         NaN
freq             1           NaN         NaN         NaN
mean           NaN  10500.000000  8200.00000  213.833333
std            NaN   1043.072385   322.49031   13.876839
min            NaN   9300.000000  7900.00000  198.000000
25%            NaN   9850.000000  8000.00000

KeyboardInterrupt: 

In [16]:

# -------------------------------
# 4. GENERATE DATA (RUN AGENT)
# -------------------------------
queries = [
    "2 + 2",
    "10 * 5",
    "100 / 4",
    "Who is Einstein?",
    "capital of France",
    "latest news",
    "3 * 9",
    "square root of 16"
]


In [17]:


print("\n--- Collecting data from agent ---\n")



--- Collecting data from agent ---



In [18]:

for q in queries:
    try:
        agent.execute_task(q)
    except:
        pass

print("Collected logs:")
print(logs)

logs.append(('2+2', 'Calculator'))

print(logs)


Collected logs:
[('fusion energy latest developments', 'Search')]
[('fusion energy latest developments', 'Search'), ('2+2', 'Calculator')]


In [19]:


# -------------------------------
# 5. TRAIN MODEL (QUERY -> TOOL)
# -------------------------------
texts = [x[0] for x in logs]
labels = [x[1] for x in logs]


In [20]:

texts


['fusion energy latest developments', '2+2']

In [21]:

labels


['Search', 'Calculator']

In [22]:

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

dtm_pd = pd.DataFrame( X.toarray(), index=texts, columns=vectorizer.get_feature_names_out()  ).head(10)
dtm_pd


,developments,energy,fusion,latest
fusion energy latest developments,0.5,0.5,0.5,0.5
2+2,0.0,0.0,0.0,0.0


In [23]:



model = LogisticRegression()
model.fit(X, labels)

print("\n--- Model trained ---\n")



--- Model trained ---



In [24]:

# -------------------------------
# 6. TRAINED POLICY (PREDICT TOOL)
# -------------------------------
def predict_tool(query):
    X = vectorizer.transform([query])
    return model.predict(X)[0]


In [25]:

# -------------------------------
# 7. FINAL AGENT (USES TRAINED POLICY)
# -------------------------------
def run_agent(query):
    chosen_tool = predict_tool(query)

    print(f"\nQuery: {query}")
    print(f"Chosen Tool (trained): {chosen_tool}")

    if chosen_tool == "Calculator":
        result = calculator_tool(query)
    else:
        result = search_tool(query)

    print(f"Result: {result}")
    return result


In [26]:

# -------------------------------
# 8. TEST
# -------------------------------
print("\n--- Testing trained agent ---")

run_agent("5 * 6")
run_agent("Who invented electricity?")
run_agent("12 + 45")
run_agent("population of Japan")




--- Testing trained agent ---

Query: 5 * 6
Chosen Tool (trained): Calculator
Result: 30

Query: Who invented electricity?
Chosen Tool (trained): Calculator
Result: Error in calculation

Query: 12 + 45
Chosen Tool (trained): Calculator
Result: 57

Query: population of Japan
Chosen Tool (trained): Calculator
Result: Error in calculation


'Error in calculation'